# Multiome BMMC ATAC imputation

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS1_S12_neurips2021_multiome_bmmc_imputation")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from adjustText import adjust_text
import matplotlib.patches as mpatches
import math
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    auc,
)


# Correlation imputed expression vs. real expression

In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_multiome_BMMC(1.0, True, True)


In [ ]:
mdata_counts["rna"].X = np.array(mdata_counts["rna"].X)
rna_norm = mdata_counts["rna"].X / mdata_counts["rna"].X.sum(axis=1, keepdims=True)


In [ ]:
from scipy.stats import rankdata

def calc_spearman_over_genes(rna_norm_filtered, imputed_expression_filtered):

    rna_norm_ranks = rankdata(rna_norm_filtered, axis=0)
    imputed_expression_ranks = rankdata(imputed_expression_filtered, axis=0)

    rna_norm_centered = rna_norm_ranks - np.mean(rna_norm_ranks, axis=0)
    imputed_expression_centered = imputed_expression_ranks - np.mean(
        imputed_expression_ranks, axis=0
    )

    covariance = np.sum(rna_norm_centered * imputed_expression_centered, axis=0)

    std_rna_norm = np.sqrt(np.sum(rna_norm_centered**2, axis=0))
    std_imputed_expression = np.sqrt(np.sum(imputed_expression_centered**2, axis=0))

    spearman_correlations = covariance / (std_rna_norm * std_imputed_expression)

    spearman_correlations = spearman_correlations[~np.isnan(spearman_correlations)]

    return spearman_correlations


In [ ]:
def plot_hexbin(rna_norm, rna_imputed, save_name):
    set_figure_params('single', 3.5)
    plt.figure()
    g = plt.hexbin(x=np.log(rna_norm + 0.0001), y=np.log(rna_imputed + 0.0001), cmap=cmap, gridsize=500, bins='log')
    plt.xlabel('log imputed')
    plt.ylabel('log observed')
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
mod_index = 0
mod_index_not_paired = 1

multivi_corr_imputed_expression = []
sparsevi_corr_imputed_expression = []
multivi_rmse_imputed_expression = []
sparsevi_rmse_imputed_expression = []
multivi_spearman_over_genes_expression = []
sparsevi_spearman_over_genes_expression = []
paired_rates = [0.8, 0.6, 0.4, 0.2]

for paired_rate in tqdm(paired_rates):

    n_paired = math.ceil(len(mdata_counts.obs) * paired_rate) + 1
    n_unpaired = math.floor((len(mdata_counts.obs) - n_paired) / len(mdata_counts.mod))
    n_paired += len(mdata.obs) - (n_paired + n_unpaired * len(mdata.mod))

    indices_unpaired = list(
        set(
            range(
                n_paired + n_unpaired * mod_index_not_paired,
                n_paired + n_unpaired * (mod_index_not_paired + 1),
            )
        )
        & set(train_indices)
    )

    imputed_expression = np.load(
        f"../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_{paired_rate}_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/default/imputed_expression.npy"
    )
    multivi_corr_imputed_expression.append(
        scipy.stats.spearmanr(
            rna_norm[indices_unpaired].flatten(), imputed_expression[indices_unpaired].flatten()
        )[0]
    )
    multivi_rmse_imputed_expression.append(
        np.sqrt(
            np.mean(
                (
                    rna_norm[indices_unpaired].flatten()
                    - imputed_expression[indices_unpaired].flatten()
                )
                ** 2
            )
        )
    )
    multivi_spearman_over_genes_expression.append(
        calc_spearman_over_genes(rna_norm[indices_unpaired], imputed_expression[indices_unpaired])
    )
    plot_hexbin(
        rna_norm[indices_unpaired].flatten(),
        imputed_expression[indices_unpaired].flatten(),
        False,
    )

    imputed_expression = np.load(
        f"../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_{paired_rate}_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/imputed_expression.npy"
    )
    sparsevi_corr_imputed_expression.append(
        scipy.stats.spearmanr(
            rna_norm[indices_unpaired].flatten(), imputed_expression[indices_unpaired].flatten()
        )[0]
    )
    sparsevi_rmse_imputed_expression.append(
        np.sqrt(
            np.mean(
                (
                    rna_norm[indices_unpaired].flatten()
                    - imputed_expression[indices_unpaired].flatten()
                )
                ** 2
            )
        )
    )
    sparsevi_spearman_over_genes_expression.append(
        calc_spearman_over_genes(rna_norm[indices_unpaired], imputed_expression[indices_unpaired])
    )
    plot_hexbin(
        rna_norm[indices_unpaired].flatten(),
        imputed_expression[indices_unpaired].flatten(),
        False,
    )


In [ ]:
paired_rates = [
    sns_paired_rate
    for r, paired_rate in zip(
        multivi_spearman_over_genes_expression + sparsevi_spearman_over_genes_expression,
        [0.8, 0.6, 0.4, 0.2] + [0.8, 0.6, 0.4, 0.2],
    )
    for sns_paired_rate in [paired_rate] * len(r)
]


In [ ]:
spearmanr_r = np.concatenate(
    multivi_spearman_over_genes_expression + sparsevi_spearman_over_genes_expression
)


In [ ]:
methods = [
    sns_method
    for r, method in zip(
        multivi_spearman_over_genes_expression + sparsevi_spearman_over_genes_expression,
        ["MultiVI"] * 4 + ["SparseVI"] * 4,
    )
    for sns_method in [method] * len(r)
]


In [ ]:
df = pd.DataFrame({'paired rate': paired_rates, 'spearman r': spearmanr_r, 'method': methods})
set_figure_params('single', 2)
fig = plt.figure()
ax = sns.boxplot(data=df, x='paired rate', y='spearman r', order=[0.8, 0.6, 0.4, 0.2], hue='method')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


In [ ]:
df = pd.DataFrame({'paired rate': paired_rates, 'spearman r': spearmanr_r, 'method': methods})
set_figure_params('single', 2)
fig = plt.figure()
ax = sns.violinplot(data=df, x='paired rate', y='spearman r', order=[0.8, 0.6, 0.4, 0.2], hue='method')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


In [ ]:
set_figure_params('single', 2)
plt.figure()
plt.plot([np.sum(sparsevi_spearman_over_genes_expression[0] > threshold) / np.sum(multivi_spearman_over_genes_expression[0] > threshold) if np.sum(sparsevi_spearman_over_genes_expression[0] > threshold) > 1 and np.sum(multivi_spearman_over_genes_expression[0] > threshold) > 1 else np.nan for threshold in np.arange(0.5, 1.0, 0.01)], label='0.8')
plt.plot([np.sum(sparsevi_spearman_over_genes_expression[1] > threshold) / np.sum(multivi_spearman_over_genes_expression[1] > threshold) if np.sum(sparsevi_spearman_over_genes_expression[1] > threshold) > 1 and np.sum(multivi_spearman_over_genes_expression[1] > threshold) > 1 else np.nan for threshold in np.arange(0.5, 1.0, 0.01)], label='0.6')
plt.plot([np.sum(sparsevi_spearman_over_genes_expression[2] > threshold) / np.sum(multivi_spearman_over_genes_expression[2] > threshold) if np.sum(sparsevi_spearman_over_genes_expression[2] > threshold) > 1 and np.sum(multivi_spearman_over_genes_expression[2] > threshold) > 1 else np.nan for threshold in np.arange(0.5, 1.0, 0.01)], label='0.4')
plt.plot([np.sum(sparsevi_spearman_over_genes_expression[3] > threshold) / np.sum(multivi_spearman_over_genes_expression[3] > threshold) if np.sum(sparsevi_spearman_over_genes_expression[3] > threshold) > 1 and np.sum(multivi_spearman_over_genes_expression[3] > threshold) > 1 else np.nan for threshold in np.arange(0.5, 1.0, 0.01)], label='0.2')
plt.xlabel('spearman r')
plt.ylabel('ratio genes imputation')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
mod_index = 1
mod_index_not_paired = 0
atac_binarized = (np.array(mdata_counts["atac"].X) > 0).astype(np.int8)

multivi_average_precision_score = []
sparsevi_average_precision_score = []
multivi_roc_auc_score = []
sparsevi_roc_auc_score = []
paired_rates = [0.8, 0.6, 0.4, 0.2]

for paired_rate in tqdm(paired_rates):
    n_paired = math.ceil(len(mdata_counts.obs) * paired_rate) + 1
    n_unpaired = math.floor((len(mdata_counts.obs) - n_paired) / len(mdata_counts.mod))
    n_paired += len(mdata.obs) - (n_paired + n_unpaired * len(mdata.mod))

    indices_unpaired = list(
        set(
            range(
                n_paired + n_unpaired * mod_index_not_paired,
                n_paired + n_unpaired * (mod_index_not_paired + 1),
            )
        )
        & set(train_indices)
    )

    accessibility_estimates = pd.read_pickle(
        f"../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_{paired_rate}_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/default/accessibility_estimates_df.pickle"
    )
    atac_imputed = accessibility_estimates.to_numpy()
    multivi_average_precision_score.append(
        average_precision_score(
            atac_binarized[indices_unpaired].flatten(), atac_imputed[indices_unpaired].flatten()
        )
    )
    multivi_roc_auc_score.append(
        roc_auc_score(
            atac_binarized[indices_unpaired].flatten(), atac_imputed[indices_unpaired].flatten()
        )
    )

    accessibility_estimates = pd.read_pickle(
        f"../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_{paired_rate}_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/accessibility_estimates_df.pickle"
    )
    atac_imputed = accessibility_estimates.to_numpy()
    sparsevi_average_precision_score.append(
        average_precision_score(
            atac_binarized[indices_unpaired].flatten(), atac_imputed[indices_unpaired].flatten()
        )
    )
    sparsevi_roc_auc_score.append(
        roc_auc_score(
            atac_binarized[indices_unpaired].flatten(), atac_imputed[indices_unpaired].flatten()
        )
    )


In [ ]:
paired_rates = [0.8, 0.6, 0.4, 0.2, 0.8, 0.6, 0.4, 0.2]
average_precision_score = multivi_average_precision_score + sparsevi_average_precision_score
methods = ['MultiVI'] * 4 + ['SparseVI'] * 4
df = pd.DataFrame({'paired rate': paired_rates, 'avg. precision score': average_precision_score, 'method': methods})
set_figure_params('single', 2)
fig = plt.figure()
ax = sns.barplot(data=df, x='paired rate', y='avg. precision score', order=[0.8, 0.6, 0.4, 0.2], hue='method')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
save_name = 'neurips2021_multiome_bmmc_atac_imputation_MultiVI_SparseVI'
plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
paired_rates = [0.8, 0.6, 0.4, 0.2, 0.8, 0.6, 0.4, 0.2]
auroc = multivi_roc_auc_score + sparsevi_roc_auc_score
methods = ["MultiVI"] * 4 + ["SparseVI"] * 4

df = pd.DataFrame({"paired rate": paired_rates, "AUROC": auroc, "method": methods})

set_figure_params("single", 2)
ax = sns.barplot(data=df, x="paired rate", y="AUROC", order=[0.8, 0.6, 0.4, 0.2], hue="method")
ax.set_ylim(0.5, None)

plt.legend(loc="upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()
